# Voyage AI SDK Demo

This notebook demonstrates two core Voyage AI capabilities:

| Part | Concept | Models Used |
|------|---------|-------------|
| **A** | Shared Embedding Space | `voyage-4` vs `voyage-4-lite` |
| **B** | Reranker Instruction Following | `rerank-2.5` |

Run each cell top-to-bottom. Estimated runtime: ~20 seconds (API calls).

## Setup
### Step 1 — Install dependencies

In [1]:
%pip install -q voyageai numpy

### Step 2 — Set your Voyage AI API key

Use **Colab Secrets** (recommended — keeps the key out of your notebook):
1. Click the **🔑 key icon** in the left sidebar
2. Add a secret named `VOYAGE_API_KEY` with your key as the value
3. Toggle **Notebook access** ON
4. Run the cell below

> Alternatively, replace the `userdata.get(...)` line with `os.environ["VOYAGE_API_KEY"] = "pa-..."` and delete the cell before sharing.

In [2]:
import os
from google.colab import userdata

os.environ["VOYAGE_API_KEY"] = userdata.get("VOYAGE_API_KEY")
print("API key loaded ✓" if os.environ.get("VOYAGE_API_KEY") else "⚠️  Key not found — check Colab Secrets")

API key loaded ✓


### Step 3 — Imports and Voyage AI client

In [3]:
import numpy as np
import voyageai

vo = voyageai.Client()

def cosine(u, v):
    """Cosine similarity between two vectors."""
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)))

print("voyageai", voyageai.__version__, "— client ready ✓")

voyageai 0.3.7 — client ready ✓


---
## Part A — Shared Embedding Space

**Claim:** All models in the `voyage-4` series (`voyage-4-large`, `voyage-4`, `voyage-4-lite`) share the **same embedding space**. This means:

- A vector from `voyage-4` and a vector from `voyage-4-lite` are **directly comparable**
- You can mix models in a retrieval pipeline (e.g. embed queries with `voyage-4-lite` for speed, index documents with `voyage-4` for quality)
- Cross-model cosine similarity scores are just as meaningful as within-model scores

We'll verify this with 3 sentences:

| Label | Sentence | Expected relationship |
|-------|----------|-----------------------|
| S1 | *A dog runs through a sunny park* | baseline |
| S2 | *A canine dashes across a bright meadow* | semantically **similar** to S1 |
| S3 | *The quarterly earnings report exceeded expectations* | semantically **unrelated** to S1 |

### Embed the same sentences with two different models

In [4]:
sentences = [
    "A dog runs through a sunny park",                      # S1
    "A canine dashes across a bright meadow",               # S2 — similar to S1
    "The quarterly earnings report exceeded expectations",  # S3 — unrelated
]

print("Calling voyage-4 ...")
emb_a = vo.embed(sentences, model="voyage-4", input_type="document").embeddings

print("Calling voyage-4-lite ...")
emb_b = vo.embed(sentences, model="voyage-4-lite", input_type="document").embeddings

print(f"\nvoyage-4      vector dimensions: {len(emb_a[0])}")
print(f"voyage-4-lite vector dimensions: {len(emb_b[0])}")

Calling voyage-4 ...
Calling voyage-4-lite ...

voyage-4      vector dimensions: 1024
voyage-4-lite vector dimensions: 1024


### Compare cosine similarity: within-model vs cross-model

For each sentence pair we compute similarity three ways:
1. **voyage-4 × voyage-4** — within model A
2. **voyage-4 × voyage-4-lite** — *cross-model* (the key column)
3. **voyage-4-lite × voyage-4-lite** — within model B

If column 2 tracks columns 1 and 3, the shared-space claim holds.

In [5]:
pairs = [
    (0, 0, "S1 vs S1  (same sentence)"),
    (0, 1, "S1 vs S2  (semantically similar)"),
    (0, 2, "S1 vs S3  (unrelated topics)"),
]

col_w = 24
print(f"{'Sentence pair':<32} {'voyage-4 × voyage-4':>{col_w}} "
      f"{'voyage-4 × voyage-4-lite':>{col_w}} {'voyage-4-lite × voyage-4-lite':>{col_w}}")
print("-" * (32 + col_w * 3 + 6))

for i, j, desc in pairs:
    within_a = cosine(emb_a[i], emb_a[j])
    cross    = cosine(emb_a[i], emb_b[j])
    within_b = cosine(emb_b[i], emb_b[j])
    print(f"{desc:<32} {within_a:>{col_w}.4f} {cross:>{col_w}.4f} {within_b:>{col_w}.4f}")

print()
print("✓ The cross-model column tracks the within-model columns.")
print("  voyage-4 series vectors are interoperable across model sizes.")

Sentence pair                         voyage-4 × voyage-4 voyage-4 × voyage-4-lite voyage-4-lite × voyage-4-lite
--------------------------------------------------------------------------------------------------------------
S1 vs S1  (same sentence)                          1.0000                   0.9581                   1.0000
S1 vs S2  (semantically similar)                   0.8729                   0.8377                   0.8590
S1 vs S3  (unrelated topics)                       0.4251                   0.4307                   0.4419

✓ The cross-model column tracks the within-model columns.
  voyage-4 series vectors are interoperable across model sizes.


---
## Part B — Reranker Instruction Following

`rerank-2.5` goes beyond relevance scoring — it accepts a **natural-language instruction** that steers *which kind* of result to prioritize, without changing the query itself.

**Scenario:** A user searches `"how to use layers"`. Without context, any layer tutorial (Photoshop, Illustrator, After Effects) could be relevant. With an instruction, we tell the reranker exactly what we want.

| | Run 1 | Run 2 |
|--|-------|-------|
| Query | `"how to use layers"` | `"how to use layers"` |
| Instruction | *(none)* | `"Prioritize beginner-friendly Photoshop tutorials only"` |

We'll use 6 candidate documents — a mix of Photoshop beginner, Illustrator advanced, and After Effects guides.

### Define the query and candidate documents

In [6]:
query = "how to use layers"

documents = [
    "Photoshop for beginners: how to create and manage layers in your first project",      # D1
    "Illustrator advanced guide: using artboards and layers for complex vector workflows", # D2 ← target
    "After Effects animation: compositing with layer-based timeline and keyframes",        # D3
    "Beginner Photoshop tutorial: understanding the Layers panel and blend modes",         # D4
    "After Effects expressions and layer parenting for advanced motion graphics",          # D5
    "Illustrator layers: organizing paths and groups for print and web projects",          # D6 ← target
]

instruction = "Prioritize Illustrator tutorials for advanced users only"

print(f"Query     : {query}")
print(f"Instruction: {instruction}")
print(f"\n{len(documents)} candidate documents loaded.")
for idx, doc in enumerate(documents, 1):
    tag = " ← target" if "Illustrator" in doc and ("advanced" in doc.lower() or "Beginner" in doc) else ""
    print(f"  D{idx}: {doc}{tag}")

Query     : how to use layers
Instruction: Prioritize Illustrator tutorials for advanced users only

6 candidate documents loaded.
  D1: Photoshop for beginners: how to create and manage layers in your first project
  D2: Illustrator advanced guide: using artboards and layers for complex vector workflows ← target
  D3: After Effects animation: compositing with layer-based timeline and keyframes
  D4: Beginner Photoshop tutorial: understanding the Layers panel and blend modes
  D5: After Effects expressions and layer parenting for advanced motion graphics
  D6: Illustrator layers: organizing paths and groups for print and web projects


### Run 1 — Rerank without instruction (default relevance)

In [7]:
res_default = vo.rerank(
    query,
    documents,
    model="rerank-2.5",
)

print("Default ranking (no instruction):")
print("-" * 80)
for rank, r in enumerate(res_default.results, 1):
    print(f"  #{rank}  [{r.relevance_score:.4f}]  {documents[r.index]}")

Default ranking (no instruction):
--------------------------------------------------------------------------------
  #1  [0.6562]  Photoshop for beginners: how to create and manage layers in your first project
  #2  [0.6367]  Beginner Photoshop tutorial: understanding the Layers panel and blend modes
  #3  [0.5508]  Illustrator advanced guide: using artboards and layers for complex vector workflows
  #4  [0.5156]  Illustrator layers: organizing paths and groups for print and web projects
  #5  [0.5039]  After Effects animation: compositing with layer-based timeline and keyframes
  #6  [0.4805]  After Effects expressions and layer parenting for advanced motion graphics


### Run 2 — Rerank with instruction

The `instruction` parameter steers the model to re-score documents through the lens of the instruction — **without changing the query**.

In [8]:
res_instruct = vo.rerank(
    query=f"{instruction}\nQuery: {query}",
    documents=documents,
    model="rerank-2.5",
)

print(f'Instruction ranking: "{instruction}"')
print("-" * 80)
for rank, r in enumerate(res_instruct.results, 1):
    print(f"  #{rank}  [{r.relevance_score:.4f}]  {documents[r.index]}")

Instruction ranking: "Prioritize Illustrator tutorials for advanced users only"
--------------------------------------------------------------------------------
  #1  [0.7461]  Illustrator advanced guide: using artboards and layers for complex vector workflows
  #2  [0.5781]  Illustrator layers: organizing paths and groups for print and web projects
  #3  [0.3789]  Beginner Photoshop tutorial: understanding the Layers panel and blend modes
  #4  [0.3789]  Photoshop for beginners: how to create and manage layers in your first project
  #5  [0.3711]  After Effects expressions and layer parenting for advanced motion graphics
  #6  [0.3418]  After Effects animation: compositing with layer-based timeline and keyframes


### Side-by-side comparison

In [9]:
rank_w = 60
print(f"{'Rank':<5} {'Default (no instruction)':<{rank_w}} Instructed")
print("-" * (5 + rank_w + 55))

for rank, (d, i) in enumerate(zip(res_default.results, res_instruct.results), 1):
    doc_d = (d.document[:rank_w - 3] + "…") if len(d.document) > rank_w - 3 else d.document
    doc_i = (i.document[:50] + "…") if len(i.document) > 50 else i.document
    print(f"#{rank:<4} {doc_d:<{rank_w}} {doc_i}")

print()
print("✓ With the instruction, Illustrator tutorials rise to the top.")
print("  Same query — different intent — different ranking.")

Rank  Default (no instruction)                                     Instructed
------------------------------------------------------------------------------------------------------------------------
#1    Photoshop for beginners: how to create and manage layers …   Illustrator advanced guide: using artboards and la…
#2    Beginner Photoshop tutorial: understanding the Layers pan…   Illustrator layers: organizing paths and groups fo…
#3    Illustrator advanced guide: using artboards and layers fo…   Beginner Photoshop tutorial: understanding the Lay…
#4    Illustrator layers: organizing paths and groups for print…   Photoshop for beginners: how to create and manage …
#5    After Effects animation: compositing with layer-based tim…   After Effects expressions and layer parenting for …
#6    After Effects expressions and layer parenting for advance…   After Effects animation: compositing with layer-ba…

✓ With the instruction, Illustrator tutorials rise to the top.
  Same query — differen